# Parts 3–4 · 01 Download

The only notebook in parts 3–4 that talks to WRDS. It saves raw pulls to `cache/` as parquet files, one per table (and per year for the big ones), and logs each pull in `cache/manifest.csv`.

**Rerun-safe:** anything already saved is skipped. If Colab disconnects, rerun the folder and connect cells, then the pull cell. Only missing years are pulled.

**Before running:** put `common.py` and `download.py` in `3_4_surprise_returns/code/`. Nothing here uses Mahaut's files.

`cache/` holds WRDS data: it stays in the team's private storage and is never part of the submission.

In [3]:
!pip install -q wrds pandas_market_calendars pyarrow
# A red pip message about ibis-framework and toolz is harmless.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.


In [4]:
import os, sys
from pathlib import Path

# Find the checkout from the repository root or any notebook subfolder.
_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts").is_dir()
                  and (p / "3_4_surprise_returns" / "code").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned S08 repository, "
                            "or set S08_ROOT to its path.")
# Optional private data root; code always comes from this checkout.
TEAM_DIR = Path(os.environ.get("TEAM_DIR", REPO_ROOT)).expanduser().resolve()
os.environ["TEAM_DIR"] = str(TEAM_DIR)
SCRIPTS = REPO_ROOT / "scripts"
NOV2_DATA = TEAM_DIR / "1_2nov" / "nov2_data"
if not NOV2_DATA.is_dir():
    NOV2_DATA = TEAM_DIR / "files csv"


PROJECT_DIR = TEAM_DIR / "3_4_surprise_returns"


os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(REPO_ROOT / "3_4_surprise_returns" / "code"))
missing = [f for f in ["common.py", "download.py"] if not (REPO_ROOT / "3_4_surprise_returns" / "code" / f).exists()]
assert not missing, f"Missing code in {PROJECT_DIR / 'code'}: {missing}"
print("project folder:", PROJECT_DIR)

Mounted at /content/drive
project folder: /content/drive/MyDrive/S08 NLP/3_4_surprise_returns


In [5]:
import wrds

WRDS_USER = input("WRDS username: ").strip()
db = wrds.Connection(wrds_username=WRDS_USER)   # answer n to the .pgpass question

WRDS username: bblank
Enter your WRDS username [bblank]:bblank
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: n
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## Core pulls

What the checkpoint needs:
- **Links:** I/B/E/S to CRSP, Compustat to CRSP, EDGAR CIK to Compustat.
- **I/B/E/S:** quarterly EPS actuals and individual estimates (unadjusted).
- **CRSP daily:** prices, opens, returns, adjustment factors and delistings.
- **Market data:** daily Fama-French factors and NYSE size breakpoints.

All years run 2005–2025. Rerunning this cell pulls only what is missing, for example the stopped estimates and I/B/E/S surprise summary added after the first run. The big pulls go one year at a time, and the printed timing of the first year tells you roughly how long the rest will take. If a line says `FAILED`, copy it to me; the other pulls carry on.

In [6]:
import importlib
import download as d
importlib.reload(d)

result = d.run(db, d.CORE, user=WRDS_USER)
print()
print(result["status"].value_counts().to_string())


status
cached    132


## Sanity checks

Quick look at what came back. Copy this cell's output to me.

In [7]:
import pandas as pd

C = PROJECT_DIR / "cache"


def load(name):
    files = [f for f in sorted((C / name).glob("*.parquet")) if ".partial" not in f.name]
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)


act = load("ibes_actuals")
t = act["anntims"].astype("string")
print(f"I/B/E/S quarterly EPS actuals: {len(act):,} | tickers: {act.ticker.nunique():,}")
print("share with a usable announcement time:", round((t.notna() & t.ne("00:00:00")).mean(), 3))
print("currencies:", act["curr_act"].value_counts().head(4).to_dict())
print("per year:", act.groupby(act["anndats"].dt.year).size().to_dict())

det = pd.read_parquet(C / "ibes_detail" / "2015.parquet")
print(f"\nestimates made in 2015: {len(det):,} | fpi: {det.fpi.value_counts().to_dict()}")
print("currencies:", det["curr"].value_counts().head(4).to_dict(),
      "| basic/diluted flag:", det["pdf"].value_counts().to_dict())

px = pd.read_parquet(C / "crsp_daily" / "2015.parquet")
print(f"\nCRSP 2015: {len(px):,} rows | {px.permno.nunique():,} stocks | {px.date.nunique()} days")
print("share of rows with an open price:", round(px["dlyopen"].notna().mean(), 3),
      "| with a return:", round(px["dlyret"].notna().mean(), 3))

ff = pd.read_parquet(C / "ff_daily.parquet")
print(f"\nFama-French daily: {ff.date.min().date()} to {ff.date.max().date()}")

bp = load("nyse_breakpoints")
print(f"NYSE breakpoints: {len(bp)} month-ends, {bp.date.min().date()} to {bp.date.max().date()}")

I/B/E/S quarterly EPS actuals: 441,824 | tickers: 12,217
share with a usable announcement time: 0.999
currencies: {'USD': 429924, 'CAD': 5347, 'CNY': 2679, 'EUR': 1880}
per year: {2005: 26341, 2006: 26712, 2007: 25688, 2008: 23950, 2009: 22089, 2010: 20836, 2011: 19688, 2012: 19566, 2013: 19712, 2014: 20343, 2015: 20362, 2016: 19748, 2017: 19485, 2018: 18932, 2019: 18628, 2020: 19075, 2021: 21038, 2022: 20719, 2023: 20304, 2024: 19666, 2025: 18942}

estimates made in 2015: 297,183 | fpi: {'6': 154157, '7': 143026}
currencies: {} | basic/diluted flag: {'D': 296538, 'P': 645}

CRSP 2015: 951,275 rows | 3,982 stocks | 252 days
share of rows with an open price: 0.972 | with a return: 0.986

Fama-French daily: 2004-01-02 to 2026-09-30
NYSE breakpoints: 252 month-ends, 2005-01-31 to 2025-12-31


## Later: wider baseline and GAAP check (optional now)

Revenue actuals and estimates, company guidance, and Compustat GAAP EPS. They aren't needed for the checkpoint, so run this whenever there's time. The `peek` line only records the columns of tables we haven't used yet (stopped estimates, the I/B/E/S surprise summary); it pulls no data.

In [8]:
schemas = d.peek(db)

ibes.stopu_epsus: ticker, cusip, oftic, cname, astpdats, estimator, pdicity, measure, usfirm, fpedats, astptims
ibes.excu_epsus: ticker, cusip, oftic, cname, actdats, estimator, analys, fpi, measure, value, excfla, usfirm, curr, fpedats, acttims, excdats, exctims, excends, excetims
ibes.surpsumu: ticker, oftic, measure, fiscalp, pyear, pmon, usfirm, anndats, actual, surpmean, surpstdev, suescore
ibes.statsumu_epsus: ticker, cusip, oftic, cname, statpers, measure, fiscalp, fpi, estflag, curcode, numest, numup, numdown, medest, meanest, stdev, highest, lowest, usfirm, fpedats
ibes.actu_xepsus: ticker, cusip, oftic, cname, pends, measure, pdicity, anndats, anntims, actdats, acttims, value, curr_act, usfirm
ibes.detu_xepsus: ticker, cusip, oftic, cname, actdats, estimator, analys, currfl, pdf, fpi, measure, value, curr, usfirm, fpedats, acttims, revdats, revtims, anndats, anntims, report_curr


In [9]:
# Why did revenue come back empty? Row counts by measure in the unadjusted and
# adjusted tables. Copy this output to me before rerunning the revenue pulls.
d.measure_counts(db)

ibes.actu_xepsus: {'GPS': np.int64(74173), 'DPS': np.int64(74156), 'CPS': np.int64(74152), 'BPS': np.int64(74142), 'EBS': np.int64(74127), 'CSH': np.int64(74120), 'FFO': np.int64(74112)}
ibes.act_xepsus: {'NET': np.int64(74181), 'EBI': np.int64(74179), 'SAL': np.int64(74179), 'PRE': np.int64(74174), 'EBT': np.int64(74173), 'GPS': np.int64(74173), 'CPX': np.int64(74164), 'ROE': np.int64(74162)}
ibes.detu_xepsus: {'GPS': np.int64(1877234), 'BPS': np.int64(542218), 'CPS': np.int64(474457), 'DPS': np.int64(318790), 'EBS': np.int64(159292), 'FFO': np.int64(120348), 'EPX': np.int64(111428), 'CSH': np.int64(4470)}
ibes.det_xepsus: {'SAL': np.int64(2748069), 'NET': np.int64(2474257), 'EBI': np.int64(2452571), 'EBT': np.int64(2081883), 'PRE': np.int64(1995947), 'GPS': np.int64(1877725), 'ENT': np.int64(1868813), 'GRM': np.int64(801087)}


In [10]:
result_later = d.run(db, d.LATER, user=WRDS_USER)
print(result_later["status"].value_counts().to_string())

retry   ibes_rev_actuals 2005: cached file is empty
ok      ibes_rev_actuals 2005          26,510 rows     1.7s
retry   ibes_rev_actuals 2006: cached file is empty
ok      ibes_rev_actuals 2006          26,522 rows     1.2s
retry   ibes_rev_actuals 2007: cached file is empty
ok      ibes_rev_actuals 2007          25,575 rows     1.0s
retry   ibes_rev_actuals 2008: cached file is empty
ok      ibes_rev_actuals 2008          23,902 rows     0.7s
retry   ibes_rev_actuals 2009: cached file is empty
ok      ibes_rev_actuals 2009          22,047 rows     0.6s
retry   ibes_rev_actuals 2010: cached file is empty
ok      ibes_rev_actuals 2010          20,810 rows     0.5s
retry   ibes_rev_actuals 2011: cached file is empty
ok      ibes_rev_actuals 2011          19,643 rows     0.5s
retry   ibes_rev_actuals 2012: cached file is empty
ok      ibes_rev_actuals 2012          19,541 rows     0.5s
retry   ibes_rev_actuals 2013: cached file is empty
ok      ibes_rev_actuals 2013          19,631 rows  

## What's in the cache

In [11]:
print(d.summary().to_string())

                       files      rows  minutes  mb_on_disk
pull                                                       
crsp_daily                21  20947867     14.2       663.2
ibes_detail               21   5536023      2.6        91.4
ibes_rev_detail           21   4552627      3.9       106.0
ibes_stopped              21   1389123      0.3         6.9
link_cik_gvkey             1   1089697      0.2        10.0
comp_fundq                 1   1052012      1.0        29.5
ibes_actuals              21    441824      0.2        10.6
ibes_rev_actuals          21    441348      0.2        12.4
ibes_surprise_summary     21    338121      0.1         8.4
ibes_guidance             21    318568      0.3         9.8
crsp_names                 1     87839      0.0         3.0
link_ccm                   1     41036      0.0         1.0
link_ibes_crsp             1     37662      0.0         0.8
crsp_delists               1     10863      0.0         0.4
ff_daily                   1      5722  

In [12]:
db.close()